# Pipeline benchmark — reviewed windows vs hand-verified truth

Runs the pipeline **unattended** on the two reviewed windows of the full-half videos, then
scores it against the hand-corrected `*_unified.csv` files (IDF1, ID switches, MOTA) plus
the no-ground-truth proxy metrics. About 3,700 frames in total; at the last measured speed
(~1 frame/s on a T4) cell 6 takes about an hour.

It also records every model output (detections, pitch keypoints, ball re-detections), times
each pipeline stage, and makes half-resolution copies of the two windows. Together these form
a **replay bundle**: tracking experiments can then be re-run without the models or a GPU.

**Before running**
1. `Runtime → Change runtime type → T4 GPU`.
2. Colab Secrets (🔑, left sidebar): `ROBOFLOW_API_KEY`.
3. In Google Drive, folder `MyDrive/Playbook/benchmark/` containing:
   - `half1.mp4`, `half2.mp4` — the **full, untrimmed** half videos the original CSVs came from
   - `per_frame_tracks_half1_unified.csv`, `per_frame_tracks_half2_unified.csv`

**Running (people or Claude in Chrome):** run cells top to bottom, once each. Cell 6 is the
long one; it skips any half that already finished, so after a disconnect just re-run from
cell 1. If a cell fails, stop and report its last output lines. Results are saved to
`MyDrive/Playbook/benchmark/results_<timestamp>/`; the replay bundle to
`MyDrive/Playbook/benchmark/replay_bundle/` (upload that folder to the GitHub repo).

In [ ]:
# Cell 1 — Verify GPU
import subprocess
g = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
                   capture_output=True, text=True)
if g.returncode != 0:
    raise SystemExit('No GPU. Runtime -> Change runtime type -> T4 GPU, then re-run.')
print('GPU:', g.stdout.strip())

In [ ]:
# Cell 2 — Install (GPU build), once. CUDA onnxruntime goes LAST so nothing shadows it.
!apt-get install -qq ffmpeg libglib2.0-0 libsm6 libxext6 libxrender-dev >/dev/null
!pip uninstall -qqy opencv-python opencv-python-headless >/dev/null 2>&1
!pip install -q \
    'numpy>=2.0.0,<2.4.0' opencv-python-headless==4.10.0.84 tqdm 'requests>=2.32.3' \
    'pydantic>=2.11.7,<2.12.0' pydantic-settings==2.4.0 python-dotenv==1.0.1 \
    'supervision==0.27.0.post2' 'ultralytics>=8.4.37,<8.5.0' 'lap>=0.5.13,<0.6' \
    'transformers>=5.2.0,<5.3.0' 'pandas>=2.0' scipy
!pip install -q git+https://github.com/roboflow/sports.git@main
!pip install -q inference-gpu==1.3.0
!pip install -q onnxruntime-gpu==1.20.1 \
    --extra-index-url https://aiinfra.pkgs.visualstudio.com/PublicPackages/_packaging/onnxruntime-cuda-12/pypi/simple/

import onnxruntime as ort
print('onnxruntime providers:', ort.get_available_providers())
assert 'CUDAExecutionProvider' in ort.get_available_providers(), 'CUDA EP missing — re-run this cell.'
print('OK — GPU inference ready.')

In [ ]:
# Cell 3 — Clone or update the repo
import os, sys
BRANCH = 'claude/setup-gpu-video-testing-JhgUH'
REPO = 'https://github.com/muwafagq/playbook-program.git'
DEST = '/content/playbook'
if os.path.isdir(DEST + '/.git'):
    !git -C {DEST} fetch -q origin {BRANCH}
    !git -C {DEST} checkout -q {BRANCH}
    !git -C {DEST} reset -q --hard origin/{BRANCH}
else:
    !git clone -q --branch {BRANCH} {REPO} {DEST}
os.chdir(DEST); sys.path.insert(0, DEST)
head = !git -C {DEST} log --oneline -1
print('HEAD:', head[0])

In [ ]:
# Cell 4 — .env = baseline.env + API key (no source files are modified)
import shutil
shutil.copy(f'{DEST}/baseline.env', f'{DEST}/.env')
from google.colab import userdata
ROBOFLOW_API_KEY = userdata.get('ROBOFLOW_API_KEY')
with open(f'{DEST}/.env', 'a') as f:
    f.write(f'\nROBOFLOW_API_KEY={ROBOFLOW_API_KEY}\n')
print('.env configured.')

In [ ]:
# Cell 5 — Mount Drive, check inputs
import os
from google.colab import drive
drive.mount('/content/drive')

DATA = '/content/drive/MyDrive/Playbook/benchmark'
HALVES = {
    'half1': {'video': f'{DATA}/half1.mp4', 'gt': f'{DATA}/per_frame_tracks_half1_unified.csv', 'window': (1066, 3333)},
    'half2': {'video': f'{DATA}/half2.mp4', 'gt': f'{DATA}/per_frame_tracks_half2_unified.csv', 'window': (4270, 5720)},
}
WORK = '/content/bench2'
os.makedirs(WORK, exist_ok=True)

missing = [p for h in HALVES.values() for p in (h['video'], h['gt']) if not os.path.exists(p)]
if missing:
    raise SystemExit('Missing in Drive:\n  ' + '\n  '.join(missing))

import cv2
for name, h in HALVES.items():
    cap = cv2.VideoCapture(h['video'])
    n, fps = int(cap.get(cv2.CAP_PROP_FRAME_COUNT)), cap.get(cv2.CAP_PROP_FPS)
    w, ht = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)), int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    cap.release()
    h['fps'] = fps
    ok = n > h['window'][1]
    print(f"{name}: {n} frames @ {fps:.2f} fps, {w}x{ht}  window {h['window']}  {'OK' if ok else 'VIDEO TOO SHORT — is this the full half?'}")
    if not ok:
        raise SystemExit(f'{name}: video ends before the window. Use the full, untrimmed half.')

In [ ]:
# Cell 6 — Run the pipeline unattended on each window, recording all model outputs
#          and per-stage timing (skips halves already done)
import subprocess, sys
env = os.environ.copy()
env.update({'DEVICE': 'cuda', 'ONNXRUNTIME_EXECUTION_PROVIDERS': 'CUDAExecutionProvider',
            'ROBOFLOW_API_KEY': ROBOFLOW_API_KEY, 'MAX_FRAMES': '0',
            # Record every detection down to 0.10 so replays can test lower thresholds, but
            # keep the effective cut-off at 0.30 for this run (same as the baseline, where
            # DET_CONF=0.30 made the lower per-class settings inactive).
            'DET_CONF': '0.10', 'DET_CONF_PLAYER': '0.30', 'DET_CONF_REFEREE': '0.30',
            'DET_CONF_GOALKEEPER': '0.30', 'DET_CONF_BALL': '0.30'})

for name, h in HALVES.items():
    out = f'{WORK}/{name}/run'
    if os.path.exists(f'{WORK}/{name}/cache/meta.json'):
        print(f'{name}: already done, skipping.')
        continue
    os.makedirs(out, exist_ok=True)
    lo, hi = h['window']
    cmd = [sys.executable, f'{DEST}/main.py', '--source-video', h['video'], '--out-dir', out,
           '--enable-team', '--start-frame', str(lo), '--end-frame', str(hi),
           '--record-cache', f'{WORK}/{name}/cache', '--no-video']
    print(f'{name}: running frames {lo}-{hi} ...', flush=True)
    with open(f'{out}/run.log', 'w') as log:
        p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, env=env)
        for line in p.stdout:
            log.write(line)
            if line.startswith(('[stage]', '[WARN]', '[timing]', 'Done', 'Processed', 'Elapsed', 'Traceback')) or 'Error' in line:
                print('  ' + line.rstrip(), flush=True)
        p.wait()
    if p.returncode != 0:
        print(open(f'{out}/run.log').read()[-3000:])
        raise SystemExit(f'{name}: pipeline FAILED (code {p.returncode}). Full log: {out}/run.log')
print('All windows processed.')

In [ ]:
# Cell 6b — Half-resolution copies of each window (frame 0 = window start), for replays.
# Exact frames via ffmpeg's select filter, then an alignment check against the source.
import numpy as np

def frame_at(path, idx):
    cap = cv2.VideoCapture(path)
    for _ in range(idx):
        cap.grab()
    ok, f = cap.read()
    cap.release()
    return f

def make_proxy(video, lo, hi, out, crf):
    vf = f"select='between(n,{lo},{hi})',setpts=N/FRAME_RATE/TB,scale=960:-2"
    base = ['ffmpeg', '-loglevel', 'error', '-y', '-i', video, '-vf', vf, '-an',
            '-c:v', 'libx264', '-preset', 'medium', '-crf', str(crf)]
    r = subprocess.run(base + ['-fps_mode', 'passthrough', out], capture_output=True, text=True)
    if r.returncode != 0:  # older ffmpeg
        r = subprocess.run(base + ['-vsync', '0', out], capture_output=True, text=True)
    if r.returncode != 0:
        raise SystemExit('ffmpeg failed: ' + r.stderr[-1500:])

for name, h in HALVES.items():
    lo, hi = h['window']
    proxy = f'{WORK}/{name}/proxy.mp4'
    crf = 24
    while True:
        make_proxy(h['video'], lo, hi, proxy, crf)
        size_mb = os.path.getsize(proxy) / 1e6
        if size_mb < 24 or crf >= 36:
            break
        crf += 3
    cap = cv2.VideoCapture(proxy)
    n = int(cap.get(cv2.CAP_PROP_FRAME_COUNT)); cap.release()
    checks = {}
    for k in (0, hi - lo):
        p = frame_at(proxy, k).astype(np.float32)
        d = {off: float(np.abs(cv2.resize(frame_at(h['video'], lo + k + off), (p.shape[1], p.shape[0]))
                               .astype(np.float32) - p).mean()) for off in (-1, 0, 1)}
        checks[k] = min(d, key=d.get)
    aligned = all(v == 0 for v in checks.values())
    print(f"{name}: proxy {n} frames (expected {hi - lo + 1}), {size_mb:.1f} MB, crf {crf}, "
          f"alignment {'OK' if aligned else 'OFF ' + str(checks)}")
    if n != hi - lo + 1 or not aligned:
        raise SystemExit(f'{name}: proxy clip does not match the window exactly — report this output.')

In [ ]:
# Cell 7 — Score: proxy metrics (pred and truth) + accuracy against truth
import pandas as pd

def bench(*args):
    r = subprocess.run([sys.executable, '-m', 'tools.benchmark', *args], cwd=DEST,
                       capture_output=True, text=True)
    if r.returncode != 0:
        print(r.stdout[-2000:], r.stderr[-3000:])
        raise SystemExit('benchmark failed: ' + ' '.join(args[:1]))

for name, h in HALVES.items():
    lo, hi = h['window']
    base, fps = f'{WORK}/{name}', str(round(h['fps'], 3))
    gt = pd.read_csv(h['gt'], low_memory=False)
    gt[(gt.frame >= lo) & (gt.frame <= hi)].to_csv(f'{base}/gt_window.csv', index=False)
    bench('run', '--csv', f'{base}/run/per_frame_tracks.csv', '--kpi', f'{base}/run/kpi_summary.json',
          '--out-dir', f'{base}/bench_pred', '--fps', fps, '--spotcheck', '0')
    bench('run', '--csv', f'{base}/gt_window.csv', '--out-dir', f'{base}/bench_truth', '--fps', fps, '--spotcheck', '0')
    for id_col, tag in (('display_track_id', 'display'), ('track_id', 'stable')):
        bench('score-gt', '--pred', f'{base}/run/per_frame_tracks.csv', '--gt', f'{base}/gt_window.csv',
              '--pred-id-col', id_col, '--out-dir', f'{base}/score_{tag}', '--fps', fps)
    print(f'{name}: scored.')

In [ ]:
# Cell 8 — Summary
import json
def J(p): return json.load(open(p))
rows = []
for name in HALVES:
    b = f'{WORK}/{name}'
    s, st = J(f'{b}/score_display/score_gt.json'), J(f'{b}/score_stable/score_gt.json')
    bp, bt = J(f'{b}/bench_pred/benchmark.json'), J(f'{b}/bench_truth/benchmark.json')
    rows.append({
        'half': name,
        'minutes': round(bp['input']['duration_min'], 2),
        'IDF1 (shown ids)': s['IDF1'], 'IDF1 (stable ids)': st['IDF1'],
        'ID switches': s['id_switches'], 'switches / player-min': s['id_switches_per_player_minute'],
        'ids covering 2+ players': s['pred_ids_covering_2plus_players'],
        'real players': s['gt_players'], 'ids used': s['pred_ids'],
        'MOTA': s['MOTA'], 'det recall': s['detection_recall'], 'det precision': s['detection_precision'],
        'image jumps / player-min (pred)': bp['jumps']['image_jumps_per_player_minute'],
        'image jumps / player-min (truth)': bt['jumps']['image_jumps_per_player_minute'],
        'homography ok': bp['homography'].get('homography_ok_rate'),
        'pitch jitter ratio (pred)': bp.get('pitch_jitter', {}).get('jitter_ratio'),
        'tracker lost player': s.get('switch_attribution', {}).get('tracker_id_changes'),
        '  stabilizer re-linked': s.get('switch_attribution', {}).get('stabilizer_bridged'),
        '  stabilizer failed': s.get('switch_attribution', {}).get('stabilizer_failed_to_bridge'),
        '  failed within 5 frames': s.get('switch_attribution', {}).get('failed_bridges_by_gap_frames', {}).get('<=5'),
        'switches caused by stabilizer': s.get('switch_attribution', {}).get('stabilizer_caused'),
        'warnings': '; '.join(s['warnings']) or '—',
    })
summary = pd.DataFrame(rows).set_index('half').T
summary.to_csv(f'{WORK}/summary.csv')
print(summary.to_string())
for name in HALVES:
    pp = pd.read_csv(f'{WORK}/{name}/score_display/gt_per_player.csv')
    print(f'\n{name} — worst-tracked players:')
    print(pp.head(6)[['gt_id', 'gt_frames', 'id_accuracy', 'id_switches', 'distinct_pred_ids']].to_string(index=False))

In [ ]:
# Cell 9 — Save results and the replay bundle to Drive
import time, shutil, glob
stamp = time.strftime('%Y%m%d_%H%M')
dest = f"{DATA}/results_{stamp}"
shutil.copytree(WORK, dest, ignore=shutil.ignore_patterns('gt_window.csv', 'cache', 'proxy.mp4'))

bundle = f"{DATA}/replay_bundle"
if os.path.exists(bundle):
    shutil.move(bundle, f"{bundle}_old_{stamp}")
for name in HALVES:
    b = f'{bundle}/{name}'
    shutil.copytree(f'{WORK}/{name}/cache', f'{b}/cache')
    shutil.copy(f'{WORK}/{name}/proxy.mp4', f'{b}/proxy.mp4')
    for fn in ('per_frame_tracks.csv', 'kpi_summary.json', 'run.log'):
        shutil.copy(f'{WORK}/{name}/run/{fn}', f'{b}/{fn}')

print('Results:', dest)
print('Replay bundle:', bundle)
for fp in sorted(glob.glob(f'{bundle}/**/*', recursive=True)):
    if os.path.isfile(fp):
        mb = os.path.getsize(fp) / 1e6
        print(f"  {fp[len(bundle)+1:]:40s} {mb:7.2f} MB{'  <-- over GitHub 25 MB web-upload limit' if mb > 25 else ''}")